# TEXAS — Google Colab Quickstart

This notebook sets up TEXAS (`texas-psm`) on Google Colab and walks through
a basic forward + inverse temperature reconstruction.

**No local installation needed** — everything runs in the cloud.

---

## Step 1 — Install TEXAS and CmdStan

Run the cell below once per Colab session (takes ~3 minutes the first time).

> **Note on installation method**
> - **Before PyPI publication** (now): upload the wheel file manually — see the cell below.
> - **After PyPI publication**: use `pip install texas-psm` directly.
> - **If the GitHub repo is public**: use `pip install git+https://github.com/PaleoLipidRR/TEXAS.git`.

In [ ]:
# ── Install TEXAS ──────────────────────────────────────────────────────────────
#
# Choose ONE of the three options below based on your situation.

# ── Option 1: PyPI  (use this after the package is published) ─────────────────
# !pip install -q texas-psm

# ── Option 2: GitHub  (use this if the repo is public, before PyPI) ───────────
# !pip install -q git+https://github.com/PaleoLipidRR/TEXAS.git

# ── Option 3: Wheel file  (use this NOW, before PyPI publication) ─────────────
# Step A — upload the wheel from your computer:
from google.colab import files
print("Select the texas_psm-*.whl file from your dist/ folder...")
uploaded = files.upload()   # opens a file picker

# Step B — install the uploaded wheel:
import os
whl = [f for f in uploaded.keys() if f.endswith(".whl")]
if whl:
    os.system(f"pip install -q '{whl[0]}'")
    print(f"✅ Installed {whl[0]}")
else:
    print("⚠ No .whl file detected. Make sure you selected the right file.")

# ── Install CmdStan (required for Bayesian sampling — always needed) ───────────
import cmdstanpy
cmdstanpy.install_cmdstan(progress=True)

print("✅ Setup complete")

---

## Step 2 — Mount Google Drive (optional)

Skip this cell if your data is uploaded directly to Colab.

After mounting, your Google Drive files are at `/content/drive/MyDrive/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Example: list files in a subfolder
import os
data_folder = "/content/drive/MyDrive/"   # ← change to your folder
print(os.listdir(data_folder))

---

## Step 3 — Load your data

Either read from Google Drive or upload a file directly to Colab.

In [ ]:
import numpy as np
import pandas as pd

# Option A — from Google Drive
# df = pd.read_csv("/content/drive/MyDrive/your-folder/coretop_data.csv")

# Option B — upload a file interactively
# from google.colab import files
# uploaded = files.upload()   # prompts a file picker
# df = pd.read_csv(list(uploaded.keys())[0])

# Option C — use example data for testing
np.random.seed(42)
n = 30
example_df = pd.DataFrame({
    "scaledRI": np.random.uniform(0.3, 0.8, n),
    "SST":      np.random.uniform(5, 30, n),
})
df = example_df
df.head()

---

## Step 4 — Load a cached posterior and predict Ring Index

If you have a pre-computed posterior `.nc` file, upload it to Colab or
read it from Google Drive. Then run the forward prediction.

In [ ]:
from TEXAS import predict_RI_from_T, load_posterior

# Load a posterior from Google Drive
# posterior = load_posterior("/content/drive/MyDrive/posteriors/gen_logi_fixed_hier_crtp_multiv_SST.nc")

# Or pass the posterior name if it's already in the default cache
# result = predict_RI_from_T(
#     temperatures=np.linspace(5, 35, 100),
#     posterior="gen_logi_fixed_hier_crtp_multiv_SST",
# )
# result["p50"]   # median calibration curve
print("Load a posterior .nc file to run forward prediction.")

---

## Step 5 — Inverse temperature reconstruction

Predict paleotemperatures from Ring Index observations.

In [ ]:
from TEXAS import predict_T_from_RI

# result = predict_T_from_RI(
#     scaledRI        = df["scaledRI"].values,
#     prior_mu_t      = 15.0,   # prior mean temperature (°C)
#     prior_sigma_t   = 10.0,   # prior uncertainty (°C)
#     fwd_posterior_name = "gen_logi_fixed_hier_crtp_multiv_SST",
#     temptype        = "SST",
# )
#
# result["p50"]   # median reconstructed temperature
# result["p5"]    # 5th percentile
# result["p95"]   # 95th percentile
print("Provide a forward posterior to run inverse reconstruction.")

---

## Step 6 — Save results back to Google Drive

In [ ]:
# Save a results DataFrame to Google Drive
# output_path = "/content/drive/MyDrive/your-folder/texas_results.csv"
# results_df = pd.DataFrame({
#     "scaledRI": df["scaledRI"].values,
#     "T_p5":  result["p5"],
#     "T_p50": result["p50"],
#     "T_p95": result["p95"],
# })
# results_df.to_csv(output_path, index=False)
# print(f"Saved to {output_path}")
print("Uncomment above to save results to Google Drive.")

---

## Notes

| | Colab | Local / Docker |
|---|---|---|
| Setup | ~3 min first run | One-time Docker build (~15 min) |
| Data access | Google Drive or file upload | Any local folder |
| Session persistence | Resets after ~12 h idle | Persistent |
| CmdStan | Re-installed each session | Pre-built in image |
| Recommended for | Quick exploration, sharing results | Full analysis runs |

**CmdStan note**: `cmdstanpy.install_cmdstan()` downloads and compiles CmdStan
fresh every Colab session (~2–3 min). This is unavoidable in the free tier.
On Colab Pro the runtime persists longer.

**Posterior files**: Forward calibration posteriors (`.nc` files) are not
bundled with the package. Store them in Google Drive and load them with
`load_posterior(path)` or the string name if they are in the default cache directory.